<a href="https://colab.research.google.com/github/AndrijaM06/car-price-prediction/blob/main/05_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Treniranje regresionih modela (Google Colab verzija)

Do sada smo istražili podatke, očistili ih, napravili nove karakteristike i
pripremili ih za model. Sada treniramo **četiri različita regresiona
algoritma** nad istim trening skupom, kako bismo u sledećem koraku
(evaluacija) mogli pošteno da ih uporedimo:

- **Linear Regression**
- **Decision Tree**
- **Random Forest**
- **Support Vector Machine (SVR)**

Svaki model koristi isti preprocessing, isti trening i test skup.

## Preuzimanje potrebnih fajlova sa GitHub-a


In [1]:
import os
import urllib.request

os.makedirs("src", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

base_url = "https://raw.githubusercontent.com/AndrijaM06/car-price-prediction/main"

files_to_download = {
    "src/data_preprocessing.py": f"{base_url}/src/data_preprocessing.py",
    "data/cars_features.csv": f"{base_url}/data/cars_features.csv",
}

for local_path, url in files_to_download.items():
    urllib.request.urlretrieve(url, local_path)
    print(f"Preuzeto: {local_path}")

Preuzeto: src/data_preprocessing.py
Preuzeto: data/cars_features.csv


## Učitavanje pripremljenog skupa podataka

In [2]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/cars_features.csv")
df = pd.read_csv(DATA_PATH)
df.shape

(55585, 17)

## Razdvajanje ulaznih karakteristika i ciljne promenljive

In [3]:
from src.data_preprocessing import split_features_and_target, build_preprocessor

X, y = split_features_and_target(df)
print(X.shape)
print(y.shape)

(55585, 15)
(55585,)


## Podela na trening i test skup

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (44468, 15) Test: (11117, 15)


## Definisanje modela koje treniramo

Koristimo rečnik `models`, gde je ključ naziv modela, a vrednost konkretan
scikit-learn algoritam. `random_state=42` koristimo kod algoritama koji
imaju element slučajnosti, kako bi rezultati bili ponovljivi.

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

models = {
    "linear_regression": LinearRegression(),
    "decision_tree": DecisionTreeRegressor(random_state=42),
    "random_forest": RandomForestRegressor(random_state=42),
    "svm": SVR(),
}

list(models.keys())

['linear_regression', 'decision_tree', 'random_forest', 'svm']

## Treniranje i čuvanje svih modela

Za svaki algoritam pravimo novi `Pipeline` (preprocessing + regressor),
treniramo ga i čuvamo u poseban `.joblib` fajl.


In [6]:
import time
import joblib
from sklearn.pipeline import Pipeline

MODELS_DIR = Path("models")

for model_name, regressor in models.items():
    print(f"--- Training: {model_name} ---")

    model = Pipeline(steps=[
        ("preprocessor", build_preprocessor()),
        ("regressor", regressor),
    ])

    start_time = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start_time
    print(f"Training took {elapsed:.1f} seconds")

    model_path = MODELS_DIR / f"{model_name}_model.joblib"
    joblib.dump(model, model_path)
    print(f"Model saved to: {model_path}\n")

print("Svi modeli su istrenirani i sačuvani.")

--- Training: linear_regression ---
Training took 0.8 seconds
Model saved to: models/linear_regression_model.joblib

--- Training: decision_tree ---
Training took 1.3 seconds
Model saved to: models/decision_tree_model.joblib

--- Training: random_forest ---
Training took 69.1 seconds
Model saved to: models/random_forest_model.joblib

--- Training: svm ---
Training took 342.3 seconds
Model saved to: models/svm_model.joblib

Svi modeli su istrenirani i sačuvani.


## Zaključak

Istrenirali smo i sačuvali četiri regresiona modela:
- `linear_regression_model.joblib`
- `decision_tree_model.joblib`
- `random_forest_model.joblib`
- `svm_model.joblib`

Svi modeli su trenirani na istom trening skupu, sa istim preprocessing
pipelineom. U sledećem koraku učitavamo sva četiri modela i formalno merimo
koliko svaki od njih greši, koristeći MAE, MSE, RMSE i R² metrike.